# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR^2 Clinicopathological and Molecular CRC dataset](https://doi.org/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema, accessible at:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant Schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("{} ({}):\n{}".format(metadata.name, metadata.version, metadata.description))

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

All entity references use their unique `@id` as required.

In [ ]:
# List all record sets with their @id and field IDs
record_sets = metadata.record_sets

print("Available record sets (@id):")
for rs in record_sets:
    print(f"  - {rs['@id']}: {rs['name'] if 'name' in rs else ''}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("    Fields (@id):")
    for f in fields:
        # Each field is a dictionary with '@id', 'name' and possibly 'column'
        print(f"      - {f['@id']}: {f.get('name', '')}")

## 3. Data Extraction
Extract data from the primary record set into a DataFrame for analysis. All entity references are via their canonical `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Each item is a dictionary mapping field @id to value
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

print("Loaded DataFrames:")
for k, v in dataframes.items():
    print(f"  - {k}: shape={v.shape}")

# Pick the first (main) record set for demonstration
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print("\nMain record set columns (@id):")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply standard processing steps - filter records, normalize numeric fields, and group by a categorical field.

All references to fields use their `@id` as per Croissant schema.

In [ ]:
# Example: Find a numeric field and a category/grouping field by @id from schema
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Find an integer/float field's @id (fallback: first column)
    numeric_field_id = None
    group_field_id = None
    # Scan fields in main record set
    main_rs = [rs for rs in metadata.record_sets if rs["@id"] == main_record_set_id][0]
    for f in main_rs['field']:
        dtype = f.get('dataType', '').lower()
        if (dtype in ['float', 'integer', 'number']) and (f['@id'] in df.columns) and (numeric_field_id is None):
            numeric_field_id = f['@id']
        if ('Sex' in f.get('name','')) and (f['@id'] in df.columns) and (group_field_id is None):
            # Example: group by Sex if present
            group_field_id = f['@id']
    # Fallbacks
    if not numeric_field_id:
        for c in df.columns:
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_field_id = c
                break
    if not group_field_id:
        for f in main_rs['field']:
            if f.get('dataType', '').lower() in ['text', 'string'] and f['@id'] in df.columns:
                group_field_id = f['@id']
                break

    print(f"Using numeric field @id: {numeric_field_id}")
    print(f"Using group field @id: {group_field_id}")

    # Apply a threshold filter
    threshold = 20
    if numeric_field_id and numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
        display(filtered_df.head())
        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} (z-score) for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Grouped analysis
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df)

## 5. Visualization

Visualize the distribution of the numeric field, and relationships between the numeric and group field.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_id and numeric_field_id in df.columns:
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field exists, show boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

- Demonstrated loading of the Clinicopathological and Molecular CRC dataset via its Croissant schema (`mlcroissant`).
- Explored record sets and fields referenced by their unique `@id`.
- Loaded data for analysis and illustrated basic preprocessing using only canonical field references.
- Generated simple visualizations of the data.

### Notes
- For further analyses, consult the Croissant schema and [dataset description](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) for field meanings, limitations, and privacy requirements.
- All entity and field lookups strictly use `@id` to ensure robust, schema-consistent data exploration.
